In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

# Assuming you have the true labels and predicted labels
# true_labels = [...]
# predicted_labels = [...]
true_labels = np.random.randint(0, 10, 100)  # Replace with actual true labels
predicted_labels = np.random.randint(0, 10, 100)  # Replace with actual predicted labels

# Generate the confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)

# Plotting
plt.figure(figsize=(3.6, 3))
sns.set(font_scale=1)  # for label size
sns.heatmap(cm/(5), annot=False, annot_kws={"size": 8}, fmt='d', 
            # set cbar to [0, 1]
            vmin=0, vmax=1,
)
# remove cbar
plt.ylabel('True Digit')
plt.xlabel('Predicted Digit')
# plt.title('Confusion Matrix for AudioMNIST')
plt.savefig('confusion_matrix.pdf', dpi=300)
plt.show()

# Reset to default matplotlib settings
plt.rcParams.update(plt.rcParamsDefault)


# Set variables

In [4]:
GROUND_TRUTH_ROOT = "vctk/"
SENSOR_ROOT = "MICEMOUSE/gen/csv"
seed = 42

## Import Needed Libraries

In [5]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torchaudio
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import *
import os
import torchinfo
import torchvision.transforms

## Import our own backend

In [6]:
import core.nn.VCTK
from core.signal.preprocess import *
from core.nn.utils import normalize, denormalize, mapToBounds, apply_wiener

## Set seed and logging level

In [7]:
import logging
# fix seed for reproducibility
np.random.seed(seed)
torch.manual_seed(seed)
logging.basicConfig(level=logging.INFO)

In [8]:
Fs = 16000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
groundTruthConfig = {
    "root": GROUND_TRUTH_ROOT,
    "download": False,
}
sensorConfig = {
    "root": SENSOR_ROOT,
    "Fs": Fs,
    "device": device,
    "resampleMethod": 'sinc'
}

paired_ds = core.nn.VCTK.PairedAudioDataset(groundTruthConfig, sensorConfig)
filtered_paired_ds = core.nn.VCTK.removeOutliers(paired_ds)
train_loader, test_loader, dev_loader = core.nn.VCTK.getAudioLoaders(filtered_paired_ds, device = device, batch_size=32, ret_labels=True)

INFO:root:Initializing PairedAudioDataset...


Removed 1387 outliers 3.16% of the dataset
Remaining 43873 samples
Train: 1129 batches = 36128 samples
Test: 166 batches = 5312 samples
Dev: 34 batches = 1088 samples


In [13]:
speaker_list =  paired_ds.gtDS._speaker_ids
utterance_list = list(set([e[1] for e in paired_ds.gtDS._sample_ids]))
nSpeakers = len(speaker_list)
nUtterances = len(utterance_list)
onehot_speaker = lambda x: torch.eye(nSpeakers)[speaker_list.index(x)]
onehot_utterance = lambda x: torch.eye(nUtterances)[utterance_list.index(x)]
# Is this a clean way to do this? Hell nah
# Is this efficient? Yes

In [37]:
from core.nn.whisperPrimitives import *
device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = AudioEncoder(n_mels=80, n_ctx=501, n_state=96, n_head = 6, n_layer=2).to(device)
flatten = nn.Flatten()
class permute (nn.Module):
    def forward(self, x):
        return x.permute(0, 2, 1)
utteranceClassifier = nn.Sequential(
    encoder,
    permute(),
    nn.AvgPool1d(50),
    nn.Flatten(),
    nn.Linear(960, 128),
    nn.ReLU(),
    nn.Linear(128, nUtterances),
).to(device)
speakerClassifier = nn.Sequential(
    encoder,
    permute(),
    nn.AvgPool1d(50),
    nn.Flatten(),
    nn.Linear(960, 256),
    nn.ReLU(),
    nn.Linear(256, nSpeakers),
).to(device)

In [11]:
import einops
tforms = core.nn.utils.buildTransforms(device = device, 
                                        Fs = Fs, 
                                        n_fft = 800, 
                                        win_length = 800,
                                        hop_length = 160,
                                        n_mels = 80,
                                        f_max_mouse = 8000,
                                        f_max_full = 8000,
                                        )
specFn, inverseSpecFn, melFilterMouse, melFilter, inverseMelFn = tforms

toMelDB = lambda batch: core.nn.utils.toMelDB(batch, specFn=specFn, mousemelfilters=melFilterMouse, fullmelfilter=melFilter)

def computeUnboundedSpec(X):
    ms = True if X.shape[1] == 2 else False
    if ms:
        X -= einops.reduce(X, "b c t -> b c ()", "mean")  # Remove mean
    else:
        X -= einops.reduce(X, "b t -> b ()", "mean")  # Remove mean
    X /= torch.std(X)                          # Normalize                                        
    X = specFn(X)                                 # Convert to mel
    X = X.abs().pow(2)                           # Power
    
    if ms:
        X = torch.einsum("bkct,fc->bkft", X, melFilterMouse)
        X = einops.rearrange(X, "b c m t -> b (m c) t", c=2)
    else:
        X = torch.einsum("bct,fc->bft", X, melFilter) # Apply mel filter
        
    X = (torch.maximum(torch.clamp(X, min=1e-10).log10(), torch.clamp(X, min=1e-10).log10().max() - 8.0) + 4.0)/ 4.0    # Convert to dB
    return X

# # The following code is used to compute the min and max values of the spectrograms

mnmn = torch.inf
mxmx = -torch.inf
for Mwav, Wwav, _, _ in dev_loader:
    Mfilt = computeUnboundedSpec(Mwav)
    M = computeUnboundedSpec(Mwav)
    W = computeUnboundedSpec(Wwav)
    mnmn = min(mnmn, M.min(), W.min(), Mfilt.min())
    mxmx = max(mxmx, M.max(), W.max(), Mfilt.max())

import einops


def computeSpec(X):
    X = computeUnboundedSpec(X)
    X = (X - mnmn) / (mxmx - mnmn)               # Map to bounds
    X = torch.clamp(X, 0, 1)                    # Clamp to [0, 1]
    X = 2 * X - 1                               # Map to [-1, 1]
    return X

def fromSpec(X):
    if X.shape[1] == 160:
        X = einops.rearrange(X, "b (m c) t -> b c m t", c=2)
        X = torch.mean(X, dim=1)
    X = (X + 1) / 2
    X = X * (mxmx - mnmn) + mnmn
    X = 10 ** (4 * X - 4)
    X = inverseMelFn(X)
    X = inverseSpecFn(X)
    return X

In [38]:
N_EPOCHS = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = speakerClassifier
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
with tqdm(total = N_EPOCHS * len(train_loader)) as pbar:
    for epoch in range(N_EPOCHS):
        for (X, _, Y, _) in train_loader:
            X = X.to(device)
            X = computeSpec(X)

            speaker_one_hot = (torch.stack([onehot_speaker(i) for i in Y]))
            Y = torch.tensor([speaker_list.index(i) for i in Y]).to(device)
            speaker_one_hot = Y.to(device)
            optimizer.zero_grad()

            output = net(X)
            loss = criterion(output, Y)
            loss.backward()
            optimizer.step()
            pbar.set_description(f"Loss: {loss.item():.4f}")
            pbar.update(1)
torch.save(net.state_dict(), "net.pt")

Loss: 1.9428:   5%|▍         | 551/11290 [07:39<2:29:22,  1.20it/s]


KeyboardInterrupt: 

In [402]:
N_EPOCHS = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = ConvNet2D(nSpeakers).to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
stft_fn_cuda = torchaudio.transforms.Spectrogram(n_fft=N_FFT, hop_length=N_HOP, power=2).to(device)
with tqdm(range(N_EPOCHS)) as pbar:
    for epoch in pbar:
        for (waveform, speaker_id, _) in train_loader:
            waveform = waveform[:, 1:, :].to(device)
            stft = stft_fn_cuda(waveform)
            speaker_one_hot = (torch.stack([onehot_speaker(i) for i in speaker_id]))
            speaker_id = torch.tensor([speaker_list.index(i) for i in speaker_id]).to(device)
            speaker_one_hot = speaker_id.to(device)
            optimizer.zero_grad()
            output = net(stft)
            loss = criterion(output, speaker_id)
            loss.backward()
            optimizer.step()
            pbar.set_description(f"Loss: {loss.item():.4f}")
torch.save(net.state_dict(), "net2.pt")

  0%|                                                                                                                                | 0/10 [00:00<?, ?it/s]


Loss: 0.0084: 100%|██████████| 10/10 [51:57<00:00, 311.76s/it]


In [403]:
N_EPOCHS = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = ConvNet2D(nUtterances).to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
stft_fn_cuda = torchaudio.transforms.Spectrogram(n_fft=N_FFT, hop_length=N_HOP, power=2).to(device)
with tqdm(range(N_EPOCHS)) as pbar:
    for epoch in pbar:
        for (waveform, _, utterance_id) in train_loader:
            waveform = waveform[:, 0, :].to(device)
            waveform = torch.stack([waveform, waveform], dim=1)
            stft = stft_fn_cuda(waveform)
            # print(stft.shape)
            utterance_id = torch.tensor([utterance_list.index(i) for i in utterance_id]).to(device)
            optimizer.zero_grad()
            output = net(stft)
            loss = criterion(output, utterance_id)
            loss.backward()
            optimizer.step()
            pbar.set_description(f"Loss: {loss.item():.4f}")
torch.save(net.state_dict(), "net3.pt")

Loss: 5.9133: 100%|██████████| 10/10 [51:17<00:00, 307.70s/it]


In [405]:
# Get Accuracy
net = ConvNet2D(nSpeakers).to(device)
net.load_state_dict(torch.load("net.pt"))
net.eval()
correct = 0
total = 0
with torch.no_grad():
    for (waveform, speaker_id, _) in test_loader:
        waveform = waveform[:, 0, :].to(device)
        waveform = torch.stack([waveform, waveform], dim=1)
        stft = stft_fn_cuda(waveform)
        speaker_id = torch.tensor([speaker_list.index(i) for i in speaker_id]).to(device)
        outputs = net(stft)
        _, predicted = torch.max(outputs.data, 1)
        total += speaker_id.size(0)
        correct += (predicted == speaker_id).sum().item()
print('Accuracy of the network on the test set: %d %%' % (100 * correct / total))

Accuracy of the network on the test set: 71 %


In [408]:
# Get Accuracy
net = ConvNet2D(nSpeakers).to(device)
net.load_state_dict(torch.load("net2.pt"))
net.eval()
correct = 0
total = 0
with torch.no_grad():
    for (waveform, speaker_id, _) in test_loader:
        waveform = waveform[:, 1:, :].to(device)
        stft = stft_fn_cuda(waveform)
        speaker_one_hot = (torch.stack([onehot_speaker(i) for i in speaker_id]))
        speaker_id = torch.tensor([speaker_list.index(i) for i in speaker_id]).to(device)
        speaker_one_hot = speaker_id.to(device)
        outputs = net(stft)
        _, predicted = torch.max(outputs.data, 1)
        total += speaker_id.size(0)
        correct += (predicted == speaker_id).sum().item()
print('Accuracy of the network on the test set: %d %%' % (100 * correct / total))

Accuracy of the network on the test set: 55 %


In [427]:


# Get top-k accuracy
k = 5
net = ConvNet2D(nUtterances).to(device)
net.load_state_dict(torch.load("net3.pt"))
net.eval()
correct = 0
total = 0
with torch.no_grad():
    for (waveform, _, utterance_id) in train_loader:
        waveform = waveform[:, 0, :].to(device)
        waveform = torch.stack([waveform, waveform], dim=1)
        stft = stft_fn_cuda(waveform)
        # print(stft.shape)
        utterance_id = torch.tensor([utterance_list.index(i) for i in utterance_id]).to(device)
        optimizer.zero_grad()
        outputs = net(stft)
        _, predicted = torch.topk(outputs.data, k=k, dim=1)
        total += utterance_id.size(0)
        correct += (predicted == utterance_id.view(-1, 1)).sum().item()
ratio3 = 100 * correct / total
print('Accuracy of the network on the test set: %d %%' % (100 * correct / total))

Accuracy of the network on the test set: 5 %


In [429]:
ratio3*nUtterances

2592.094259684878

In [ ]:
N_EPOCHS = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = ConvNet2D(nUtterances).to(device)
net.load_state_dict(torch.load("net3.pt"))
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
stft_fn_cuda = torchaudio.transforms.Spectrogram(n_fft=N_FFT, hop_length=N_HOP, power=2).to(device)
with tqdm(range(N_EPOCHS)) as pbar:
    for epoch in pbar:
        for (waveform, _, utterance_id) in train_loader:
            waveform = waveform[:, 1:, :].to(device)
            stft = stft_fn_cuda(waveform)
            utterance_id = torch.tensor([utterance_list.index(i) for i in utterance_id]).to(device)
            optimizer.zero_grad()
            output = net(stft)
            loss = criterion(output, utterance_id)
            loss.backward()
            optimizer.step()
            pbar.set_description(f"Loss: {loss.item():.4f}")
torch.save(net.state_dict(), "net4.pt")

In [430]:
# Get Accuracy
net = ConvNet2D(nUtterances).to(device)
k = 5
net.load_state_dict(torch.load("net4.pt"))
net.eval()
correct = 0
total = 0
with torch.no_grad():
    for (waveform, _, utterance_id) in test_loader:
        waveform = waveform[:, 1:, :].to(device)
        stft = stft_fn_cuda(waveform)
        # print(stft.shape)
        utterance_id = torch.tensor([utterance_list.index(i) for i in utterance_id]).to(device)
        optimizer.zero_grad()
        outputs = net(stft)
        _, predicted = torch.topk(outputs.data, k=k, dim=1)
        total += speaker_id.size(0)
        correct += (predicted == utterance_id.view(-1, 1)).sum().item()
ratio4 = 100 * correct / total
print('Accuracy of the network on the test set: %d %%' % (100 * correct / total))

Accuracy of the network on the test set: 3 %


In [431]:
nUtterances * ratio4

1804.9937264742784

In [426]:
0.03 / (correct / total)

3.2310810810810806

In [94]:
# save the models

torch.save(net.state_dict(), "convnet2d.pt")
torch.save(net_ds2.state_dict(), "convnet2d_ds2.pt")

In [95]:
accuracy = 0 
with torch.no_grad():
    for (waveform, _, _, speaker_id, _) in tqdm(test_loader):
        waveform = waveform.to(device)
        waveform = torch.cat([waveform, waveform], dim=1)
        speaker_id = torch.tensor([speaker_list.index(i) for i in speaker_id]).to(device)
        output = net(waveform)
        accuracy += (output.argmax(1) == speaker_id).sum().item()
print(f"Accuracy: {accuracy / len(test_loader.dataset) * 100:.2f}%")

  0%|          | 0/84 [00:00<?, ?it/s]

100%|██████████| 84/84 [00:08<00:00,  9.84it/s]

Accuracy: 93.67%


In [96]:
accuracy = 0 
with torch.no_grad():
    for (waveform, _, _, speaker_id, _) in tqdm(test_ds2_loader):
        waveform = waveform.to(device)
        speaker_id = torch.tensor([speaker_list.index(i) for i in speaker_id]).to(device)
        output = net_ds2(waveform)
        accuracy += (output.argmax(1) == speaker_id).sum().item()
print(f"Accuracy: {accuracy / len(test_ds2_loader.dataset) * 100:.2f}%")

100%|██████████| 84/84 [00:10<00:00,  8.26it/s]

Accuracy: 89.05%
